# Fourier Descriptors: Curve Approximation in Frequency Space

A **closed planar curve** $\gamma : [0,1] \to \mathbb{C}$ (parameterized on the circle) can be expanded in the Fourier basis:
$$
\gamma(t) = \sum_{k=-\infty}^{+\infty} c_k\, e^{2\pi i k t}, \qquad c_k = \int_0^1 \gamma(t)\, e^{-2\pi i k t}\, dt.
$$
The **Fourier descriptors** $c_k$ capture the shape of the curve at different scales. Truncating to $|k| \leq r$ gives a low-frequency (smooth) approximation; increasing $r$ progressively reveals finer geometric details.

## Shape in frequency space

The meaning of individual Fourier modes is elegant:
- $c_0 = $ centroid of the curve.
- $|c_1|$ controls the overall size; $\arg(c_1)$ is the initial phase.
- Higher $|c_k|$ correspond to curvature oscillations at scale $1/k$.

## Bandlimited reconstruction

Given samples $\gamma(j/n)$, $j = 0,\ldots,n-1$ (computed from image contours), the DFT gives $c_k \approx \hat{\gamma}_k$. The **bandlimited reconstruction** with $r$ modes is:
$$
\gamma_r(t) = \sum_{|k| \leq r} c_k\, e^{2\pi i k t}.
$$
As $r$ increases from $1$ to $n/2$, the reconstruction evolves from an ellipse (only the $k = \pm 1$ modes) to the full detailed shape. The first few modes capture the coarse topology; higher modes add corners and sharp features.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from scipy.interpolate import interp1d

plt.rcParams['figure.dpi'] = 120

## Synthetic curves

We build closed curves programmatically: a star, a petal shape, and a more complex silhouette. Each curve is arc-length resampled to $p$ equally-spaced points before computing the DFT.

In [ ]:
def resample_curve(gamma, p=512):
    """Arc-length resample a closed curve to p points."""
    gamma = np.append(gamma, gamma[0])  # close the curve
    diffs = np.diff(gamma)
    arc = np.concatenate([[0], np.cumsum(np.abs(diffs))])
    arc /= arc[-1]
    interp_r = interp1d(arc, gamma.real)
    interp_i = interp1d(arc, gamma.imag)
    t = np.linspace(0, 1, p, endpoint=False)
    return interp_r(t) + 1j * interp_i(t)


t = np.linspace(0, 2*np.pi, 2000, endpoint=False)

curves = {}
# Star: radius oscillates
r_star = 1 + 0.4 * np.cos(5 * t)
curves['star (5 petals)'] = r_star * np.exp(1j * t)

# Trefoil-like
r_trefoil = 1 + 0.5 * np.cos(3 * t)
curves['trefoil'] = r_trefoil * np.exp(1j * t)

# Ellipse + higher harmonics
curves['ellipse+harmonics'] = (1.5 * np.cos(t) + 0.3 * np.cos(3*t)
                               + 1j * (np.sin(t) + 0.2 * np.sin(2*t)))

# Fourier-reconstruct each curve
p = 512
curves_resampled = {name: resample_curve(c, p) for name, c in curves.items()}

fig, axes = plt.subplots(1, len(curves), figsize=(11, 4))
for ax, (name, c) in zip(axes, curves_resampled.items()):
    ax.plot(c.real, c.imag, 'b-', lw=2)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(name, fontsize=10)
fig.suptitle('Closed curves for Fourier descriptor analysis', y=1.02)
plt.tight_layout()
plt.show()

## Bandlimited reconstruction at increasing resolution

We apply the DFT to the arc-length resampled star curve and reconstruct it using only the lowest $r$ Fourier modes (i.e. $|k| \leq r$). The sequence from $r=1$ (ellipse) to $r=40$ (faithful star) illustrates the progressive enrichment of geometric detail.

In [ ]:
def fourier_reconstruct(c, r):
    """Reconstruct curve using only modes |k| <= r."""
    n = len(c)
    cf = np.fft.fft(c)
    freqs = np.fft.fftfreq(n) * n  # integer frequencies
    mask = np.abs(freqs) <= r
    return np.fft.ifft(cf * mask)


curve = curves_resampled['star (5 petals)']
r_values = [1, 2, 5, 10, 20, 40]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, r in zip(axes.ravel(), r_values):
    c_r = fourier_reconstruct(curve, r)
    t_norm = r / 40
    col = (t_norm, 0, 1 - t_norm)
    ax.plot(curve.real, curve.imag, 'gray', lw=1, alpha=0.4, label='original')
    ax.plot(c_r.real, c_r.imag, lw=2.5, color=col, label=f'$r={r}$ modes')
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(f'$r = {r}$ modes  (${2*r+1}$ coefs)', fontsize=10)
    ax.legend(fontsize=8)
fig.suptitle('Fourier descriptor reconstruction: star curve', y=1.02)
plt.tight_layout()
plt.show()

## Power spectrum of curves

The **power spectrum** $|c_k|^2$ of a curve encodes how much energy is at each frequency. Smooth curves decay quickly; curves with corners or cusps have slow decay. The star's spectrum has strong peaks at $k = \pm 5$ (the 5-fold symmetry).

In [ ]:
fig, axes = plt.subplots(1, len(curves_resampled), figsize=(12, 4))
for ax, (name, c) in zip(axes, curves_resampled.items()):
    cf = np.fft.fft(c) / len(c)
    freqs = np.fft.fftfreq(len(c)) * len(c)
    idx = np.argsort(freqs)
    f_sorted = freqs[idx]
    p_sorted = np.abs(cf[idx])**2
    # show only |k| <= 30
    mask = np.abs(f_sorted) <= 30
    ax.semilogy(f_sorted[mask], p_sorted[mask] + 1e-12, 'b-o', ms=3, lw=1.5)
    ax.set_xlabel('frequency $k$'); ax.set_title(name, fontsize=9)
    ax.grid(alpha=0.3); ax.set_ylabel('$|c_k|^2$')
fig.suptitle('Power spectrum $|c_k|^2$ of closed curves', y=1.02)
plt.tight_layout()
plt.show()

## Interactive: adjust bandwidth

Slide the maximum frequency $r$ to watch the curve reconstruction evolve. Switch between the three curve types to compare how their frequency content differs.

In [ ]:
from ipywidgets import Dropdown

def show_reconstruct(curve_name='star (5 petals)', r=10):
    c = curves_resampled[curve_name]
    c_r = fourier_reconstruct(c, r)
    cf = np.fft.fft(c) / len(c)
    freqs = np.fft.fftfreq(len(c)) * len(c)
    idx = np.argsort(freqs)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].plot(c.real, c.imag, 'gray', lw=1.5, alpha=0.5, label='original')
    t_norm = r / 50
    axes[0].plot(c_r.real, c_r.imag, lw=2.5, color=(t_norm, 0, 1-t_norm),
                 label=f'$r={r}$')
    axes[0].set_aspect('equal'); axes[0].axis('off')
    axes[0].set_title(f'Reconstruction with $r={r}$ modes')
    axes[0].legend(fontsize=9)

    f_s = freqs[idx]; p_s = np.abs(cf[idx])**2
    mask = np.abs(f_s) <= 60
    axes[1].semilogy(f_s[mask], p_s[mask]+1e-12, 'b-', lw=1.5)
    axes[1].axvspan(-r, r, alpha=0.15, color='tomato', label='used modes')
    axes[1].set_xlabel('frequency $k$'); axes[1].set_ylabel('$|c_k|^2$')
    axes[1].set_title('Power spectrum'); axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_reconstruct,
         curve_name=Dropdown(options=list(curves_resampled.keys()), description='curve'),
         r=IntSlider(value=10, min=1, max=50, step=1, description='bandwidth $r$'));

## Bibliographical resources

- Zahn, C. T. and Roskies, R. Z. (1972). Fourier descriptors for plane closed curves. *IEEE Transactions on Computers*, C-21(3), 269–281.
- Persoon, E. and Fu, K.-S. (1977). Shape discrimination using Fourier descriptors. *IEEE Transactions on Systems, Man, and Cybernetics*, 7(3), 170–179.
- Oppenheim, A. V. and Schafer, R. W. (2009). *Discrete-Time Signal Processing* (3rd ed.). Prentice Hall.
- Zhang, D. and Lu, G. (2004). Review of shape representation and description techniques. *Pattern Recognition*, 37(1), 1–19.
- Dieleman, S., Willett, K. W. and Dambre, J. (2015). Rotation-invariant convolutional neural networks for galaxy morphology prediction. *MNRAS*, 450(2), 1441–1459.